# 🏋️ Step 2: ペルソナ別 PPO 学習 → ONNX 生成

```
persona_rewards.json (Step1 で生成)
  ↓
ペルソナ A〜E を順番に PPO 学習 (各 ~60M steps)
  ↓
persona_A.onnx 〜 persona_E.onnx を単一ファイルで生成
  ↓
マイドライブ / mesa_persona_onnx / に保存
  ↓
step3_persona_city_sim.html で読み込む
```

## 修正済み問題
| 問題 | 対策 |
|------|------|
| `ModuleNotFoundError: onnxscript` | `onnxscript` をインストールに追加 |
| `LayerNormalization` opset 変換失敗 | `opset_version=17` に変更 |
| training mode 警告 | `model.eval()` + `torch.no_grad()` |
| `.onnx.data` 外部ファイル問題 | `onnx.save(save_as_external_data=False)` |
| GPU 使用率が低い | N_ENVS=4096, ROLLOUT=128, MINIBATCH=16384 に最適化 |
| 旋回が1意思決定=200°になり建物間で詰まる | 1意思決定の合計旋回を ROT_DECISION_DEG=40° に固定し 4°/tick に分割 |
| ゴール報酬・距離シェイピングが観測不能(位置が見えない)でノイズ化 | compass(相対方位+距離) を観測に追加、シェイピングをペルソナ別係数化 |
| 経路の記憶がなく同じ場所を巡回 | 前後左右セクタの訪問済み率(半径5) を観測に追加 → explore/revisit が学習可能に |
| social_bonus が学習不能(相手が見えない) | パートナーの相対方位×近接度を観測に追加 |
| building_bonus が step() で未使用 | 建物セル初回進入ボーナスとして実装 |

**Runtime: T4 GPU を選択してください**

---

## 📁 Google Drive ディレクトリ構成

```
MyDrive/
├── mesa_textures/                      ← 建物テクスチャ素材 (TEX_DIR)
│   └── gyudon / ramen / bento / cafe / office / house / conbini / hospital .png (計8枚)
│
├── mesa_persona/                       ← 学習の入出力 (SAVE_DIR)
│   ├── persona_rewards.json            ← ペルソナ別 報酬パラメータ (入力)
│   └── ckpt_A.pt 〜 ckpt_E.pt           ← 学習チェックポイント (再開用の中間状態)
│
└── mesa_persona_onnx/                  ← ONNX 成果物 (ONNX_DIR)
    ├── dinov2_vits14.onnx              ← 共有DINOv2バックボーン (本NBのセル9で生成)
    ├── building_classifier.onnx (+_meta.json) ← 建物分類ヘッド (別NBで生成)
    ├── seg_head.onnx (+_meta.json)     ← セグ判定ヘッド (別NB・任意)
    └── persona_A.onnx 〜 persona_E.onnx (+_meta.json) ← 学習の最終成果物
```

### 🗑 再学習のたびに削除すべきもの

| ファイル | 理由 |
|---|---|
| `mesa_persona/ckpt_A.pt`〜`ckpt_E.pt` | **必ず削除してから再学習すること。** 同名で残っていると「続きから学習」として自動再開してしまう。入力次元が変わる変更(384→392など)は次元不一致で自動的に新規学習へ切り替わるが、**次元が同じロジック変更(報酬の重み・スケールなど)は気づかず古い重みを引き継いでしまう**ため要注意(旧バージョンで「活動量が減った」原因もこれ)。 |

### ♻️ 再学習しても消さなくてよいもの (毎回自動で上書きされる)

| ファイル | 理由 |
|---|---|
| `mesa_persona_onnx/persona_A.onnx`〜`persona_E.onnx`、`_meta.json` | `export_persona_onnx()` が毎回同名で上書き保存する。ただし大きな仕様変更(報酬構造やgoal条件付けの有無など)をした直後は、「今のファイルがどのコードで作られたか」が分かりにくくなるので手動で消してクリーンにしておくと迷子にならない。 |

### 🔒 永続・触らないもの (入力素材・共有アセット)

| ファイル | 理由 |
|---|---|
| `mesa_textures/*.png` | 建物テクスチャ。学習前に一度揃えれば以降は不変。 |
| `mesa_persona/persona_rewards.json` | ペルソナ別報酬設定。手動編集するか、本ノートブックの生成セルで作り直す(上書きされる点に注意)。 |
| `mesa_persona_onnx/dinov2_vits14.onnx` | 全ペルソナ共有のDINOv2バックボーン(frozen)。`DINO_MODEL` を変えない限り作り直し不要。 |
| `mesa_persona_onnx/building_classifier.onnx`(+meta) | 建物分類ヘッド。別ノートブック(step1.5)で生成。 |
| `mesa_persona_onnx/seg_head.onnx`(+meta) | セグメンテーション判定ヘッド(任意)。別ノートブック(step1.6)で生成。 |


In [ ]:
# ============================================================
# セル 1 置き換え — インストール (DINOv2追加)
# ============================================================
!pip install torch torchvision onnx onnxscript onnxruntime numpy -q
print('✓ Install complete')

In [ ]:
# ============================================================
# セル 2: インポート & GPU 確認 & Google Drive マウント
# ============================================================
import torch, torch.nn as nn, onnx
import numpy as np
import json, random, math, time, os, subprocess
from IPython.display import clear_output
import matplotlib.pyplot as plt, matplotlib

matplotlib.rcParams.update({
    'figure.facecolor': '#0a0c10', 'axes.facecolor': '#12161e',
    'text.color': '#c8d8e8',       'axes.labelcolor': '#c8d8e8',
    'xtick.color': '#4a5a6a',      'ytick.color': '#4a5a6a',
})

assert torch.cuda.is_available(), '❌ GPU 未検出。ランタイム → T4 GPU に変更してください'
DEVICE = torch.device('cuda')
print(f'GPU : {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

def get_gpu_stats():
    try:
        r = subprocess.run(
            ['nvidia-smi',
             '--query-gpu=utilization.gpu,memory.used,memory.total',
             '--format=csv,noheader,nounits'],
            capture_output=True, text=True, timeout=2)
        v = r.stdout.strip().split(', ')
        return float(v[0]), float(v[1])/1024, float(v[2])/1024
    except:
        return 0.0, 0.0, 15.0

from google.colab import drive
drive.mount('/content/drive')

SAVE_DIR = '/content/drive/MyDrive/mesa_persona'       # persona_rewards.json の場所
ONNX_DIR = '/content/drive/MyDrive/mesa_persona_onnx'  # ONNX 出力先
os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(ONNX_DIR, exist_ok=True)
print(f'\n入力: {SAVE_DIR}')
print(f'出力: {ONNX_DIR}')

In [ ]:
# ============================================================
# セル 3 置き換え — 定数 (DINOv2版)
# ============================================================
OTHER=0; ROAD=1; BUILDING=2; TREE=3
GRID = 30

N_ENVS    = 4096   # A100 80GB 向け。OOMなら 2048/1024 に下げる
ROLLOUT   = 64     # そのまま(増やすなら128でオンポリシー量2倍)
MINIBATCH = 65536  # B=N_ENVS*ROLLOUT に対し数ミニバッチになる程度
B         = N_ENVS * ROLLOUT

MAX_STEPS = 400
EPOCHS    = 4
CLIP_EPS  = 0.2
GAE_LAM   = 0.95
GAMMA     = 0.99
ENT_COEF  = 0.03
VF_COEF   = 0.5
LR        = 3e-4   # DINOv2 frozen の場合は少し大きめでOK
STEPS_PER_PERSONA = 4_000_000   # ★時間/品質ダイヤル: 目安 sps×1800≈30分ぶん

MOVE_DIST = 0.25

# ── Sticky action (frame skip) ──
# 本番(server.js / standalone index.html)は INFER_EVERY ティックごとに 1 回だけ
# 推論し、同じ行動をそのティック数ぶん繰り返し適用する(前進は毎ティック MOVE_DIST、
# 旋回は毎ティック ROT_RAD)。学習側も同じ回数だけ行動を繰り返して 1 意思決定と
# することで train/deploy のダイナミクスを一致させ、「旋回が効きすぎてクルクル回る」
# 不自然さを根から無くす。
# ★ server.js / index.html の INFER_EVERY と必ず同じ値にすること。
ACTION_REPEAT = 10

# ── 旋回量 ──
# 旧版: ROT_RAD=20°/tick × ACTION_REPEAT(10) = 1意思決定で200°旋回。
# 到達可能な向きが実質40°刻みに量子化され、1セル幅の道路と平行に走れず
# 「建物の間に詰まる」主因だった。1意思決定の合計旋回を ROT_DECISION_DEG に
# 固定し、ティックあたりは 1/ACTION_REPEAT (=4°/tick) を適用する。
# ★ server.js / index.html は meta の rot_per_tick_deg を読んで同じ値を使う。
ROT_DECISION_DEG = 40.0
ROT_RAD = math.radians(ROT_DECISION_DEG) / ACTION_REPEAT   # 4°/tick

# ── FPV画像設定 (DINOv2版: 224×224) ──
IMG_W   = 224   # DINOv2 の標準入力サイズ
IMG_H   = 224
IMG_CH  = 3
OBS_DIM = IMG_CH * IMG_H * IMG_W   # 224*224*3 = 150528
ACT_DIM = 3

# ── DINOv2 設定 ──
DINO_MODEL   = 'dinov2_vits14'   # vits14=384次元 / vitb14=768次元
DINO_DIM     = 384               # vits14の出力次元
# DINO_MODEL = 'dinov2_vitb14'   # より高精度版
# DINO_DIM   = 768

# DINOv2 の正規化パラメータ (ImageNet統計)
DINO_MEAN = [0.485, 0.456, 0.406]
DINO_STD  = [0.229, 0.224, 0.225]

# レイキャスト設定 (FPV画像生成用)
RAY_MAX   = 8.0
N_RAYS    = IMG_W        # 224本
RAY_STEP  = 0.1
FOV       = math.radians(60.0)

# セルタイプの RGB色
CELL_RGB = torch.tensor([
    [ 12,  30,  74],
    [176, 180, 172],
    [196,  32,  32],
    [ 35, 104,  40],
], dtype=torch.float32, device='cpu') / 255.0
SKY_RGB   = torch.tensor([ 6, 12, 20], dtype=torch.float32) / 255.0
FLOOR_RGB = torch.tensor([26, 40, 32], dtype=torch.float32) / 255.0

RAY_ANGLES_T = torch.linspace(-FOV/2, FOV/2, IMG_W, dtype=torch.float32)
RAY_DISTS_T  = torch.arange(RAY_STEP, RAY_MAX+RAY_STEP, RAY_STEP, dtype=torch.float32)
N_STEPS_RAY  = len(RAY_DISTS_T)

print(f'N_ENVS:{N_ENVS}  B:{B:,}')
print(f'FP画像: {IMG_W}×{IMG_H}×{IMG_CH}ch (DINOv2標準サイズ)')
print(f'DINOv2: {DINO_MODEL}  出力次元: {DINO_DIM}')
print(f'バッファVRAM見積: {ROLLOUT*N_ENVS*OBS_DIM*4/1e9:.2f}GB')
print()
print('⚠️  224×224はバッファが大きい。VRAMが足りない場合:')
print('   N_ENVS=1024, ROLLOUT=64 に下げてください')

# ── 方法A・案2: 観測 = CLS(384) + 建物クラス one-hot(8) = 392 ──
N_BLDG_CLASSES = 8   # gyudon..hospital (cell7で分類器metaから上書きされ得る)
OBS_FEAT     = DINO_DIM + N_BLDG_CLASSES
BLDG_CLASSES = ['gyudon','ramen','bento','cafe','office','house','conbini','hospital']
FOOD_CLASSES = [0,1,2,3]   # gyudon, ramen, bento, cafe
DINO_CHUNK   = 4096  # 1回のDINOv2 forwardのバッチ(大きいほどGPU稼働↑/VRAM↑)。N_ENVSと同値でutil最大
torch.backends.cudnn.benchmark = True   # conv高速化


In [ ]:
# ============================================================
# セル 4: 有機的マップ生成 (Domain Randomization対応)
# ============================================================
from collections import deque

def make_map_organic(size=GRID, seed=None):
    """
    有機的都市マップ生成。
    seed=None のとき毎回違うマップ (Domain Randomization用)
    seed=42 など固定値のとき再現可能な固定マップ

    アルゴリズム:
      ① 格子道路を敷く
      ② ブロック内に建物・木を配置
      ③ 道路の区間セルをランダムに削除
         (道路ネットワークの連結性をBFSで保証)
      ④ 削除セルをTREE/OTHERで埋める
    """
    rng = random.Random(seed)  # seed=None → システム時刻でランダム

    grid = np.full((size, size), OTHER, dtype=np.int32)
    step = 4
    rows = list(range(0, size, step))
    cols = list(range(0, size, step))

    # ① 格子道路
    for r in rows: grid[r, :] = ROAD
    for c in cols: grid[:, c] = ROAD

    # ② ブロック内に建物・木を配置
    for ri in range(len(rows)-1):
        for ci in range(len(cols)-1):
            r0, r1 = rows[ri]+1, rows[ri+1]
            c0, c1 = cols[ci]+1, cols[ci+1]
            cells = [(r,c) for r in range(r0,r1) for c in range(c0,c1)]
            if not cells: continue
            b = rng.choice(cells)
            grid[b[0], b[1]] = BUILDING
            for r, c in cells:
                if (r,c) == b: continue
                v = rng.random()
                if   v < 0.25: grid[r,c] = TREE
                elif v < 0.45: grid[r,c] = BUILDING

    # 道路を最終確定 (建物が侵食しないように)
    for r in rows: grid[r, :] = ROAD
    for c in cols: grid[:, c] = ROAD

    # ③ 道路の連結性チェック関数
    #    「全道路セルが1つの連結成分か」を BFS で確認
    def road_connected(g):
        start = next(
            ((r,c) for r in range(size) for c in range(size) if g[r,c]==ROAD),
            None)
        if start is None: return True
        visited = set()
        q = deque([start])
        visited.add(start)
        while q:
            r, c = q.popleft()
            for dr, dc in [(-1,0),(1,0),(0,-1),(0,1)]:
                nr, nc = r+dr, c+dc
                if 0<=nr<size and 0<=nc<size and (nr,nc) not in visited \
                   and g[nr,nc]==ROAD:
                    visited.add((nr,nc))
                    q.append((nr,nc))
        # 全道路セルが訪問済みか
        return all(
            (r,c) in visited
            for r in range(size) for c in range(size)
            if g[r,c]==ROAD
        )

    # ③ 削除候補: 交差点以外の道路区間セル
    row_set = set(rows)
    col_set = set(cols)
    candidates = [
        (r, c)
        for r in range(size) for c in range(size)
        if grid[r,c] == ROAD
        and not (r in row_set and c in col_set)  # 交差点は除外
    ]
    rng.shuffle(candidates)

    # 削除率: 25〜50% をランダムに決定
    remove_ratio = 0.25 + rng.random() * 0.25
    max_remove   = int(len(candidates) * remove_ratio)
    removed      = 0

    for r, c in candidates:
        if removed >= max_remove: break
        grid[r, c] = OTHER  # 試しに削除
        if road_connected(grid):
            # 削除確定: TREE か OTHER で埋める
            grid[r, c] = TREE if rng.random() < 0.4 else OTHER
            removed += 1
        else:
            grid[r, c] = ROAD  # 連結が切れる → 復元

    return grid


# ── 初期マップ生成 (固定seed=42) ──
MAP_NP    = make_map_organic(seed=42)
MAP_T     = torch.tensor(MAP_NP, dtype=torch.long,  device=DEVICE)
MAP_F     = MAP_T.float()
PASS_T    = (MAP_T == ROAD) | (MAP_T == BUILDING)
BCELLS_NP = np.array([(r,c) for r in range(GRID)
                             for c in range(GRID) if MAP_NP[r,c]==BUILDING])
BCELLS_T  = torch.tensor(BCELLS_NP, dtype=torch.float32, device=DEVICE)
NB        = len(BCELLS_NP)

# ── 建物サブタイプmap (各建物セルに 0-7 を割当) ──
def assign_building_types(bcells_np, seed=123):
    rng = np.random.default_rng(seed)
    bt = np.full((GRID, GRID), -1, dtype=np.int64)
    for (r, c) in bcells_np:
        bt[r, c] = int(rng.integers(0, 8))
    return bt

BTYPE_NP = assign_building_types(BCELLS_NP)
BTYPE_T  = torch.tensor(BTYPE_NP, device=DEVICE)

# ── マップ更新関数 (Domain Randomization用) ──
def randomize_map():
    """
    新しいランダムマップを生成して GPU テンソルを更新。
    PersonaVecEnv.step() の done 処理と組み合わせて呼ぶ。
    """
    global MAP_NP, MAP_T, MAP_F, PASS_T, BCELLS_NP, BCELLS_T, NB, BTYPE_NP, BTYPE_T
    MAP_NP    = make_map_organic(seed=None)  # 毎回違うマップ
    MAP_T     = torch.tensor(MAP_NP, dtype=torch.long,  device=DEVICE)
    MAP_F     = MAP_T.float()
    PASS_T    = (MAP_T == ROAD) | (MAP_T == BUILDING)
    BCELLS_NP = np.array([(r,c) for r in range(GRID)
                                 for c in range(GRID) if MAP_NP[r,c]==BUILDING])
    BCELLS_T  = torch.tensor(BCELLS_NP, dtype=torch.float32, device=DEVICE)
    NB        = len(BCELLS_NP)
    BTYPE_NP  = assign_building_types(BCELLS_NP, seed=random.randint(0, 10**9))
    BTYPE_T   = torch.tensor(BTYPE_NP, device=DEVICE)


# ── 初期マップを可視化 ──
cmap = matplotlib.colors.ListedColormap(['#1a3a6a','#d0d0c8','#c42020','#256b28'])
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.patch.set_facecolor('#0a0c10')
for i, ax in enumerate(axes):
    test_map = make_map_organic(seed=i*100 if i > 0 else 42)
    ax.imshow(test_map, cmap=cmap, vmin=0, vmax=3, origin='upper')
    roads = int((test_map==ROAD).sum())
    bldgs = int((test_map==BUILDING).sum())
    ax.set_title(
        f'seed={42 if i==0 else i*100}  roads:{roads}  bldgs:{bldgs}',
        color='#c8d8e8', fontsize=9)
    ax.axis('off')
plt.suptitle('有機的マップのサンプル (3パターン)', color='#00ddb4', fontsize=12)
plt.tight_layout()
plt.show()

print(f'初期マップ: Buildings={NB}  MAP_T={MAP_T.device}')
print(f'有機的マップ生成 ✓  randomize_map() で学習中に更新可能')

In [ ]:
# ============================================================
# セル 5 置き換え — テクスチャ付きFPVレンダリング (DDAレイキャスタ・GPU一括)
# 学習でもテクスチャを見る → DINOv2が建物種類を見分けられる
# ============================================================
from PIL import Image
import glob

TW = TH = 64
TEX_NAMES = ['gyudon','ramen','bento','cafe','office','house','conbini','hospital']  # index=建物type
TEX_DIR = '/content/drive/MyDrive/mesa_textures'   # ← ここに8枚の *.png をアップロード

def _load_tex(name):
    hits = glob.glob(f'{TEX_DIR}/**/{name}.png', recursive=True)
    assert hits, f'{name}.png が {TEX_DIR} に見つかりません (textures をアップロードしてください)'
    a = np.asarray(Image.open(hits[0]).convert('RGB').resize((TW, TH)), np.float32) / 255
    return a

TEX_T = torch.tensor(np.stack([_load_tex(n) for n in TEX_NAMES]), device=DEVICE)  # (8,TH,TW,3)
print(f'✓ テクスチャ {TEX_T.shape[0]} 枚ロード {tuple(TEX_T.shape)}')

SKY_T = torch.tensor([6,12,20],  device=DEVICE, dtype=torch.float32)/255
FLR_T = torch.tensor([26,40,32], device=DEVICE, dtype=torch.float32)/255
# 旧フラット版が参照していたグローバル (呼び出し互換のためダミー保持)
CELL_RGB_GPU = CELL_RGB.to(DEVICE) if 'CELL_RGB' in globals() else None
SKY_RGB_GPU = SKY_T; FLOOR_RGB_GPU = FLR_T; RAY_ANGLES_GPU = None; RAY_DISTS_GPU = None

def render_fp_batch(xs, ys, ths, *_, MAXSTEPS=64):
    """テクスチャ付きDDAレイキャスタ。returns (N,3,H,W) float[0,1]。
       余分な位置引数(*_)は旧シグネチャ互換で無視。建物種類は BTYPE_T を参照。"""
    N = xs.shape[0]; W = IMG_W; H = IMG_H
    cam = 2*torch.arange(W, device=DEVICE, dtype=torch.float32)/W - 1
    dirX = torch.cos(ths); dirY = torch.sin(ths)
    pl = math.tan(FOV/2); planeX = -dirY*pl; planeY = dirX*pl
    rdx = dirX[:,None] + planeX[:,None]*cam[None,:]
    rdy = dirY[:,None] + planeY[:,None]*cam[None,:]
    mapX = torch.floor(xs).long()[:,None].expand(N,W).clone()
    mapY = torch.floor(ys).long()[:,None].expand(N,W).clone()
    ddx = torch.where(rdx==0, torch.full_like(rdx,1e30), (1/rdx).abs())
    ddy = torch.where(rdy==0, torch.full_like(rdy,1e30), (1/rdy).abs())
    stepX = torch.where(rdx<0, -1, 1); stepY = torch.where(rdy<0, -1, 1)
    pX = xs[:,None]; pY = ys[:,None]
    sdx = torch.where(rdx<0, (pX-mapX)*ddx, (mapX+1-pX)*ddx)
    sdy = torch.where(rdy<0, (pY-mapY)*ddy, (mapY+1-pY)*ddy)
    hitT = torch.full((N,W), -1, dtype=torch.long, device=DEVICE)
    side = torch.zeros((N,W), dtype=torch.long, device=DEVICE)
    active = torch.ones((N,W), dtype=torch.bool, device=DEVICE)
    for _i in range(MAXSTEPS):
        xlt = sdx < sdy; sx = active & xlt; sy = active & (~xlt)
        sdx = torch.where(sx, sdx+ddx, sdx); mapX = torch.where(sx, mapX+stepX, mapX); side = torch.where(sx, 0, side)
        sdy = torch.where(sy, sdy+ddy, sdy); mapY = torch.where(sy, mapY+stepY, mapY); side = torch.where(sy, 1, side)
        inb = (mapX>=0)&(mapX<GRID)&(mapY>=0)&(mapY<GRID)
        t = BTYPE_T[mapX.clamp(0,GRID-1), mapY.clamp(0,GRID-1)]
        hitnow = active & inb & (t>=0)
        hitT = torch.where(hitnow, t, hitT)
        active = active & (~hitnow) & inb
    perp = torch.where(side==0, sdx-ddx, sdy-ddy).clamp(min=1e-4)
    lineH = H/perp
    ds = (-lineH/2+H/2).clamp(0,H-1); de = (lineH/2+H/2).clamp(0,H-1)
    wallX = torch.where(side==0, pY+perp*rdy, pX+perp*rdx); wallX = wallX - torch.floor(wallX)
    texXi = (wallX*TW).long().clamp(0,TW-1)
    flip = ((side==0)&(rdx>0))|((side==1)&(rdy<0)); texXi = torch.where(flip, TW-1-texXi, texXi)
    br = (1-perp/9).clamp(0.35,1)
    rows = torch.arange(H, device=DEVICE)[None,:,None]
    dsb = ds[:,None,:]; lhb = lineH[:,None,:]
    inwall = (rows>=dsb)&(rows<=de[:,None,:])
    texYi = ((rows-dsb)/lhb.clamp(min=1e-6)*TH).long().clamp(0,TH-1)
    typeHW  = hitT.clamp(min=0)[:,None,:].expand(N,H,W)
    texXiHW = texXi[:,None,:].expand(N,H,W)
    wallcol = TEX_T[typeHW, texYi, texXiHW] * br[:,None,:,None]
    sf = torch.where((torch.arange(H,device=DEVICE)<H/2)[:,None], SKY_T, FLR_T)
    bg = sf[None,:,None,:].expand(N,H,W,3)
    mask = (inwall & (hitT>=0)[:,None,:]).unsqueeze(-1)
    img = torch.where(mask, wallcol, bg)
    return img.permute(0,3,1,2).contiguous()

# ── サンプル確認 (テクスチャが見えるか目視) ──
_bak = BTYPE_T.clone()
for _mx,_ty in {2:0,3:1,4:2,5:3,6:4,7:5}.items():
    if 0<=_mx<GRID and 4<GRID: BTYPE_T[_mx,4]=_ty
try:
    _xs=torch.tensor([4.5],device=DEVICE); _ys=torch.tensor([0.6],device=DEVICE); _ts=torch.tensor([math.pi/2],device=DEVICE)
    _s = render_fp_batch(_xs,_ys,_ts).cpu()[0].permute(1,2,0).numpy()
    plt.figure(figsize=(6,3)); plt.imshow(_s); plt.axis('off'); plt.title('textured FPV sample'); plt.show()
finally:
    BTYPE_T = _bak
print('✓ テクスチャ付き render_fp_batch 準備完了')


In [ ]:
# ============================================================
# セル 6 置き換え — PersonaVecEnv (FP画像観測版)
# ============================================================
def building_in_view(xs, ys, ths):
    """中央レイで最初に当たる建物の種類を one-hot(N,8) で返す (完全ベクトル化)。
       道路は通過、木/空地で遮蔽されたら建物なし。"""
    N  = xs.shape[0]
    dx = torch.cos(ths); dy = torch.sin(ths)                          # (N,)
    ds = torch.arange(RAY_STEP, RAY_MAX + 1e-6, RAY_STEP, device=DEVICE)  # (S,)
    px = xs[:,None] + dx[:,None]*ds[None,:]                            # (N,S)
    py = ys[:,None] + dy[:,None]*ds[None,:]
    inb = (px>=0)&(px<GRID)&(py>=0)&(py<GRID)
    ri = px.long().clamp(0, GRID-1); ci = py.long().clamp(0, GRID-1)
    ct = torch.where(inb, MAP_T[ri, ci], torch.full_like(ri, ROAD))    # 範囲外=ROAD
    is_wall = ct != ROAD                                               # (N,S)
    has   = is_wall.any(dim=1)                                         # (N,)
    first = torch.argmax(is_wall.long(), dim=1)                        # 最初の非道路
    r_hit = ri.gather(1, first[:,None]).squeeze(1)
    c_hit = ci.gather(1, first[:,None]).squeeze(1)
    ct_hit = ct.gather(1, first[:,None]).squeeze(1)
    is_bldg = has & (ct_hit == BUILDING)
    onehot = torch.zeros(N, N_BLDG_CLASSES, device=DEVICE)
    if bool(is_bldg.any()):
        idx = torch.nonzero(is_bldg).squeeze(1)
        bt  = BTYPE_T[r_hit, c_hit].clamp(min=0)
        onehot[idx, bt[idx]] = 1.0
    return onehot


class PersonaVecEnv:
    def __init__(self, n, rp, policy_ref=None):
        self.n=n; self.rp=rp
        self.policy_ref = policy_ref   # セグメンテーション用ポリシー参照
        self._last_obs  = None         # 直前の観測キャッシュ
        self.x    =torch.zeros(n,device=DEVICE); self.y    =torch.zeros(n,device=DEVICE)
        self.th   =torch.zeros(n,device=DEVICE)
        self.gx   =torch.zeros(n,device=DEVICE); self.gy   =torch.zeros(n,device=DEVICE)
        self.pdist=torch.zeros(n,device=DEVICE)
        self.stall=torch.zeros(n,dtype=torch.long,device=DEVICE)
        self.scnt =torch.zeros(n,dtype=torch.long,device=DEVICE)
        self.px   =torch.zeros(n,device=DEVICE); self.py   =torch.zeros(n,device=DEVICE)
        self.visited=torch.zeros(n,GRID,GRID,dtype=torch.uint8,device=DEVICE)
        self._rst(torch.ones(n,dtype=torch.bool,device=DEVICE))

    def _rand_b(self,mask):
        idx=torch.randint(0,NB,(self.n,),device=DEVICE)
        return BCELLS_T[idx,0]+0.5, BCELLS_T[idx,1]+0.5

    def _dist(self): return torch.sqrt((self.x-self.gx)**2+(self.y-self.gy)**2)

    def _rst(self,mask):
        bx,by=self._rand_b(mask); gx,gy=self._rand_b(mask)
        th=torch.rand(self.n,device=DEVICE)*math.pi*2
        nd=torch.sqrt((bx-gx)**2+(by-gy)**2)
        self.x    =torch.where(mask,bx,self.x);   self.y    =torch.where(mask,by,self.y)
        self.th   =torch.where(mask,th,self.th)
        self.gx   =torch.where(mask,gx,self.gx);  self.gy   =torch.where(mask,gy,self.gy)
        self.pdist=torch.where(mask,nd,self.pdist)
        self.stall=torch.where(mask,torch.zeros_like(self.stall),self.stall)
        self.scnt =torch.where(mask,torch.zeros_like(self.scnt), self.scnt)
        self.px   =torch.where(mask,bx,self.px);  self.py   =torch.where(mask,by,self.py)
        self.visited[mask]=0

    def reset_all(self):
        self._rst(torch.ones(self.n,dtype=torch.bool,device=DEVICE))
        obs = self._obs()
        self._last_obs = obs   # キャッシュ更新
        return obs

    def _obs(self):
        """FP画像を GPU で一括生成。returns: (N, 3, H, W)"""
        return render_fp_batch(
            self.x, self.y, self.th,
            MAP_T, CELL_RGB_GPU, SKY_RGB_GPU, FLOOR_RGB_GPU,
            RAY_ANGLES_GPU, RAY_DISTS_GPU,
            IMG_W, IMG_H, GRID, RAY_MAX
        )

    def bldg(self):
        """現在状態の視界中央の建物タイプ one-hot (N,8)"""
        return building_in_view(self.x, self.y, self.th)

    def step(self, actions):
        rp  = self.rp
        self.px = self.x.clone(); self.py = self.y.clone()   # 意思決定前の位置(stall判定用)

        # 前方通行可否は 1 意思決定につき 1 回だけ判定 (本番の seg キャッシュ挙動と一致)
        if USE_SEG_PASSABLE and getattr(self, '_last_obs', None) is not None:
            seg_pass = torch.tensor(
                self.policy_ref.get_seg_passable(self._last_obs), device=DEVICE)  # (N,) bool
        else:
            seg_pass = None

        env_idx = torch.arange(self.n, device=DEVICE)
        fwd = (actions == 0)

        # ── Sticky action: 同じ行動を ACTION_REPEAT ティック適用 (本番 INFER_EVERY と一致) ──
        # ★ 重要: 毎ティック発生する報酬は rew_tick に貯め、最後に ACTION_REPEAT で割って
        #   「1 ティックぶんの平均」に正規化する。こうしないと frame skip 化で罰則(特に
        #   壁衝突)が最大 10 倍に膨らみ、方策が前進を極端に嫌って活動量が激減する。
        #   平均化により persona_rewards.json で調整済みの報酬スケール(1 step=1 tick 前提)
        #   がそのまま保たれる。
        rew_tick    = torch.zeros(self.n, device=DEVICE)
        blocked_any = torch.zeros(self.n, dtype=torch.bool, device=DEVICE)
        for _rep in range(ACTION_REPEAT):
            rew_tick = rew_tick - rp['step_penalty']
            rot = torch.where(actions == 1, torch.full_like(self.th, -ROT_RAD),
                  torch.where(actions == 2, torch.full_like(self.th,  ROT_RAD),
                              torch.zeros_like(self.th)))
            self.th = (self.th + rot) % (math.pi * 2)
            nx = (self.x + torch.cos(self.th) * MOVE_DIST).clamp(0.01, GRID - 0.01)
            ny = (self.y + torch.sin(self.th) * MOVE_DIST).clamp(0.01, GRID - 0.01)
            ri = nx.long().clamp(0, GRID - 1); ci = ny.long().clamp(0, GRID - 1)
            passable = seg_pass if seg_pass is not None else PASS_T[ri, ci]
            can_move = fwd & passable; blocked = fwd & ~passable
            blocked_any = blocked_any | blocked
            self.x = torch.where(can_move, nx, self.x)
            self.y = torch.where(can_move, ny, self.y)
            ri2 = self.x.long().clamp(0, GRID - 1); ci2 = self.y.long().clamp(0, GRID - 1)
            cur_cell = MAP_T[ri2, ci2]
            # 人間らしい歩容: 道路歩行を加点 / 建物突っ切りを減点 / 前進を促し / 旋回に小コスト
            rew_tick = torch.where(can_move & (cur_cell == ROAD), rew_tick + rp['road_bonus'],      rew_tick)
            rew_tick = torch.where(cur_cell == BUILDING,          rew_tick - rp['offroad_penalty'], rew_tick)
            rew_tick = torch.where(fwd,                           rew_tick + rp['forward_bias'],    rew_tick)
            rew_tick = torch.where(actions != 0,                  rew_tick - rp['turn_penalty'],    rew_tick)
            was = self.visited[env_idx, ri2, ci2].bool()
            rew_tick = torch.where(can_move & ~was, rew_tick + rp['explore_bonus'],   rew_tick)
            rew_tick = torch.where(can_move &  was, rew_tick - rp['revisit_penalty'], rew_tick)
            # 建物セルへの初回進入ボーナス (観光型/社交型の building_bonus をここで消費。
            # 旧版はこのパラメータがどこからも参照されていなかった)
            rew_tick = torch.where(can_move & ~was & (cur_cell == BUILDING),
                                   rew_tick + rp.get('building_bonus', 0.0), rew_tick)
            self.visited[env_idx, ri2, ci2] = 1
            if rp['social_bonus'] > 0:
                ox = torch.roll(self.x, 1); oy = torch.roll(self.y, 1)
                near = (torch.sqrt((self.x - ox)**2 + (self.y - oy)**2) < 2.0) & can_move
                rew_tick = torch.where(near, rew_tick + rp['social_bonus'], rew_tick)

        # 毎ティック報酬を平均化 (1 ティックぶんのスケールに戻す)
        rew = rew_tick / ACTION_REPEAT
        # 壁衝突ペナルティは「この意思決定で 1 回でもぶつかったか」で 1 回だけ課す
        # (10 ティック連続衝突で -10×violation になり前進を過剰に嫌うのを防ぐ)
        rew = torch.where(blocked_any, rew - rp['violation_penalty'], rew)

        # ── 1 意思決定ぶんの後処理 (stall / goal shaping / 到着判定) ──
        moved = (torch.abs(self.x - self.px) + torch.abs(self.y - self.py)) > 0.05
        self.stall = torch.where(moved, torch.zeros_like(self.stall), self.stall + 1)
        rew = torch.where(self.stall > 3, rew - rp['stall_penalty'], rew)
        ri2 = self.x.long().clamp(0, GRID - 1); ci2 = self.y.long().clamp(0, GRID - 1)
        cur_cell = MAP_T[ri2, ci2]
        nd = self._dist()
        # 目的地への接近シェイピング (potential-based)。
        # 旧版の固定 +0.5/-0.2 は全ペルソナ同一で性格差を薄めていた上に、
        # 目的地の位置が観測に無く学習不能ノイズだった (現在は aux の compass で観測可能)。
        rew = rew + rp.get('approach_coef', 0.5) * (self.pdist - nd)
        self.pdist = nd
        arrived = (cur_cell == BUILDING) & (nd < 0.8)
        rew = torch.where(arrived, rew + rp['goal_reward'], rew)
        # ── グルメ報酬: 飲食店タイプに到着で追加ボーナス ──
        arr_bt  = BTYPE_T[ri2, ci2]
        is_food = torch.zeros_like(arrived)
        for fc in rp.get('food_classes', []):
            is_food = is_food | (arr_bt == int(fc))
        rew = torch.where(arrived & is_food, rew + float(rp.get('food_bonus', 0.0)), rew)
        ngx, ngy = self._rand_b(arrived)
        self.gx = torch.where(arrived, ngx, self.gx); self.gy = torch.where(arrived, ngy, self.gy)
        self.pdist = torch.where(arrived,
            torch.sqrt((self.x - self.gx)**2 + (self.y - self.gy)**2), self.pdist)
        self.scnt += 1
        done = self.scnt >= MAX_STEPS
        self._rst(done)
        obs = self._obs()
        self._last_obs = obs   # 次意思決定の seg 判定用
        return obs, rew, done

print('PersonaVecEnv (FP画像版) ✓')

In [ ]:
# ============================================================
# セル 7 置き換え — DINOv2 PolicyNet
# ============================================================
import torchvision.transforms as T

# DINOv2 を torch.hub からロード
print(f'DINOv2 ({DINO_MODEL}) をロード中...')
dino_backbone = torch.hub.load(
    'facebookresearch/dinov2',
    DINO_MODEL,
    pretrained=True,
)
dino_backbone = dino_backbone.to(DEVICE)
dino_backbone.eval()

# 全パラメータを frozen (勾配なし)
for param in dino_backbone.parameters():
    param.requires_grad = False

n_dino = sum(p.numel() for p in dino_backbone.parameters())
print(f'✓ DINOv2 loaded: {n_dino:,} params (全frozen)')

# ImageNet 正規化 (GPU上で実行)
dino_mean = torch.tensor(DINO_MEAN, device=DEVICE).view(1,3,1,1)
dino_std  = torch.tensor(DINO_STD,  device=DEVICE).view(1,3,1,1)

def normalize_for_dino(imgs: torch.Tensor) -> torch.Tensor:
    """
    FPV画像 [0,1] を DINOv2 用に ImageNet 正規化する。
    imgs: (N, 3, H, W)  float32  [0, 1]
    returns: (N, 3, H, W) 正規化済み
    """
    return (imgs - dino_mean) / dino_std


# ── 建物分類ヘッドを ONNX から読み込む ──
import onnxruntime as ort_clf

CLASSIFIER_PATH = f'{ONNX_DIR}/building_classifier.onnx'
CLASSIFIER_META  = f'{ONNX_DIR}/building_classifier_meta.json'

if os.path.exists(CLASSIFIER_PATH):
    clf_sess = ort_clf.InferenceSession(
        CLASSIFIER_PATH, providers=['CPUExecutionProvider'])
    with open(CLASSIFIER_META) as f:
        clf_meta = json.load(f)
    N_BLDG_CLASSES = clf_meta['n_classes']
    BLDG_CLASSES   = clf_meta['classes']
    print(f'✓ 建物分類ヘッド読み込み: {N_BLDG_CLASSES}クラス  精度={clf_meta.get("best_val_acc",0):.1%}')
    print(f'  クラス: {BLDG_CLASSES}')
else:
    # 分類ヘッドなし → ゼロベクトルで代替
    clf_sess       = None
    N_BLDG_CLASSES = 8
    BLDG_CLASSES   = ['gyudon','ramen','bento','cafe','office','house','conbini','hospital']
    print('⚠ building_classifier.onnx が見つかりません')
    print('  step1_5_building_classifier.ipynb を実行してください')
    print('  → 分類スコアなしで学習を続けます')


def classify_buildings_batch(feats_np: np.ndarray) -> np.ndarray:
    """
    DINOv2 特徴量から建物クラス確率を計算 (CPU・バッチ処理)
    feats_np: (N, DINO_DIM)
    returns:  (N, N_BLDG_CLASSES)  softmax確率
    """
    if clf_sess is None:
        return np.zeros((len(feats_np), N_BLDG_CLASSES), dtype=np.float32)
    logits = clf_sess.run(None, {'dino_feat': feats_np})[0]  # (N, C)
    exp    = np.exp(logits - logits.max(axis=1, keepdims=True))
    return (exp / exp.sum(axis=1, keepdims=True)).astype(np.float32)


# ── セグメンテーションヘッドを ONNX から読み込む ──
SEG_HEAD_PATH = f'{ONNX_DIR}/seg_head.onnx'
SEG_META_PATH = f'{ONNX_DIR}/seg_head_meta.json'

if os.path.exists(SEG_HEAD_PATH):
    seg_sess = ort_clf.InferenceSession(
        SEG_HEAD_PATH, providers=['CPUExecutionProvider'])
    with open(SEG_META_PATH) as f:
        seg_meta = json.load(f)
    SEG_OPEN_CLASS = seg_meta['open_class_id']   # 2 = open (道路方向)
    N_SEG_CLASSES  = seg_meta['n_classes']        # 5
    N_PATCHES_SEG  = seg_meta['n_patches']        # 256
    print(f'✓ seg_head読み込み: {N_SEG_CLASSES}クラス  mIoU={seg_meta.get("best_miou",0):.3f}')
    print(f'  open_class_id={SEG_OPEN_CLASS} ({seg_meta["classes"][SEG_OPEN_CLASS]})')
    USE_SEG_PASSABLE = True
else:
    seg_sess         = None
    SEG_OPEN_CLASS   = 2
    N_SEG_CLASSES    = 5
    N_PATCHES_SEG    = 256
    USE_SEG_PASSABLE = False
    print('⚠ seg_head.onnx が見つかりません')
    print('  step1_6_seg_head.ipynb を実行してください')
    print('  → 移動判定は従来のマップ配列を使用します')


def get_passable_from_seg(patch_tokens_np: np.ndarray) -> np.ndarray:
    """
    DINOv2 Patch tokens からセグメンテーション予測を行い
    前方中央ピクセルが open クラスか判定する。

    patch_tokens_np: (N, N_PATCHES, DINO_DIM) float32
    returns: passable (N,) bool  True=前進可能
    """
    if seg_sess is None:
        return np.ones(len(patch_tokens_np), dtype=bool)   # フォールバック: 全て可

    # seg_head.onnx で予測 (N, N_SEG, H, W)
    seg_logits = seg_sess.run(
        None, {'patch_tokens': patch_tokens_np})[0]        # (N, 5, 224, 224)

    # 前方中央ピクセルのクラスを取得
    # FPV画像の中央下気味 (進行方向が画面中央)
    cy = seg_logits.shape[2] // 2   # H/2 = 112
    cx = seg_logits.shape[3] // 2   # W/2 = 112
    center_cls = seg_logits[:, :, cy, cx].argmax(axis=1)   # (N,) int

    return (center_cls == SEG_OPEN_CLASS)   # True = open = 前進可能


class PolicyNet(nn.Module):
    """
    DINOv2 + 建物分類ヘッド を組み込んだ Actor-Critic。

    入力: (N, 3, 224, 224) FPV画像  または  (N, 3*224*224) フラット
    処理:
      DINOv2 (frozen)         → CLS token (N, DINO_DIM=384)
      BuildingClassifier(onnx)→ クラス確率 (N, N_BLDG_CLASSES=8)
      concat                  → (N, 384+8=392)
      FC ヘッド               → (N, 256)
      Actor/Critic            → logits(N,3) / value(N,1)

    「今見えている建物が何屋か」という情報がポリシーに伝わる。
    目的クラス(牛丼屋)を報酬関数で指定すれば、
    その建物を優先して訪れるポリシーが学習される。
    """
    def __init__(self):
        super().__init__()
        # FC ヘッド (DINOv2 + 分類スコア → 256)
        self.fc = nn.Sequential(
            nn.Linear(DINO_DIM, 256),
            nn.LayerNorm(256),
            nn.ReLU(),
            nn.Linear(256, 256),
            nn.ReLU(),
        )
        self.actor  = nn.Linear(256, ACT_DIM)
        self.critic = nn.Linear(256, 1)

    def _encode(self, x: torch.Tensor):
        """
        FPV画像 → (DINOv2特徴, 建物分類スコア, patch_tokens)
        returns:
          feat         (N, DINO_DIM) GPU tensor
          bldg_probs   (N, N_BLDG_CLASSES) numpy
          patch_tokens (N, N_PATCHES, DINO_DIM) numpy  ← セグメンテーション用
        """
        if x.dim() == 2:
            x = x.view(-1, IMG_CH, IMG_H, IMG_W)
        x = normalize_for_dino(x)
        with torch.no_grad():
            out          = dino_backbone.forward_features(x)
            feat         = out['x_norm_clstoken']        # (N, 384) CLS token
            patch_tokens = out['x_norm_patchtokens']     # (N, 256, 384) Patch tokens
        feats_np       = feat.cpu().numpy().astype(np.float32)
        patch_np       = patch_tokens.cpu().numpy().astype(np.float32)
        bldg_probs     = classify_buildings_batch(feats_np)
        return feat, bldg_probs, patch_np

    def _cls(self, x: torch.Tensor):
        if x.dim() == 2: x = x.view(-1, IMG_CH, IMG_H, IMG_W)
        x = normalize_for_dino(x)
        with torch.no_grad():
            feat = dino_backbone.forward_features(x)['x_norm_clstoken']   # (N,384)
        return feat

    def forward(self, x: torch.Tensor):
        h = self.fc(self._cls(x))                       # CLS(384) → 256
        return self.actor(h), self.critic(h)

    def act(self, x: torch.Tensor):
        lg, val = self.forward(x)
        dist    = torch.distributions.Categorical(torch.softmax(lg, -1))
        a       = dist.sample()
        return a, dist.log_prob(a), dist.entropy(), val.squeeze(-1)

    # ── frozen backbone 向け: 特徴キャッシュ用 ──
    def feat(self, x: torch.Tensor):
        """画像 → DINOv2 CLS(384)。fp16 autocast + チャンクで高速・省メモリ。"""
        if x.dim() == 2: x = x.view(-1, IMG_CH, IMG_H, IMG_W)
        outs = []
        with torch.no_grad(), torch.autocast('cuda', dtype=torch.float16):
            for i in range(0, x.shape[0], DINO_CHUNK):
                xx = normalize_for_dino(x[i:i+DINO_CHUNK])
                outs.append(dino_backbone.forward_features(xx)['x_norm_clstoken'].float())
        return torch.cat(outs, 0)

    def head(self, feat: torch.Tensor):
        """CLS特徴(384) → logits/value (学習対象のFC部のみ)"""
        h = self.fc(feat)
        return self.actor(h), self.critic(h)

    def act_feat(self, feat: torch.Tensor):
        lg, val = self.head(feat)
        dist    = torch.distributions.Categorical(torch.softmax(lg, -1))
        a       = dist.sample()
        return a, dist.log_prob(a), dist.entropy(), val.squeeze(-1)

    def get_building_probs(self, x: torch.Tensor) -> np.ndarray:
        """現在の視野の建物分類確率を取得 (ログ用)"""
        _, bldg_probs, _ = self._encode(x)
        return bldg_probs

    def get_seg_passable(self, x: torch.Tensor) -> np.ndarray:
        """FPV画像から前方が通行可能か判定 (分類器は呼ばない・チャンク処理)"""
        if x.dim() == 2: x = x.view(-1, IMG_CH, IMG_H, IMG_W)
        outs = []
        with torch.no_grad():
            for i in range(0, x.shape[0], DINO_CHUNK):
                xx = normalize_for_dino(x[i:i+DINO_CHUNK])
                patch = dino_backbone.forward_features(xx)['x_norm_patchtokens']
                outs.append(patch.cpu().numpy().astype(np.float32))
        return get_passable_from_seg(np.concatenate(outs, 0))


# ── 動作確認 ──
n_fc = sum(p.numel() for p in PolicyNet().parameters())
print(f'PolicyNet FC部分 : {n_fc:,} params (これだけ学習)')
print(f'DINOv2 frozen    : {n_dino:,} params')
print(f'分類ヘッド       : {"あり" if clf_sess else "なし (ゼロ代替)"}')
print()

# 推論テスト
_net   = PolicyNet().to(DEVICE)
_dummy = torch.randn(2, IMG_CH, IMG_H, IMG_W, device=DEVICE)
_lg, _val = _net.forward(_dummy)
assert _lg.shape == (2, ACT_DIM)
print(f'forward OK: logits={_lg.shape}  val={_val.shape}')

# 建物分類スコアのテスト
_bldg = _net.get_building_probs(_dummy)
print(f'building_probs: {_bldg.shape}')
print(f'  {dict(zip(BLDG_CLASSES, _bldg[0].round(3)))}')

_pass = _net.get_seg_passable(_dummy)
print(f'seg_passable: {_pass}  (True=前方通行可能)')
print(f'  USE_SEG_PASSABLE={USE_SEG_PASSABLE}')

util, used, total = get_gpu_stats()
print(f'VRAM: {used:.2f}/{total:.1f}GB')
del _net, _dummy, _lg, _val, _bldg, _pass
print('✓ PolicyNet DINOv2+分類+セグ版 OK')

# ── 学習中は移動判定をマップ配列にして高速化 (seg_head は本番サーバーでのみ使用) ──
USE_SEG_PASSABLE = False
print(f'[train] USE_SEG_PASSABLE={USE_SEG_PASSABLE}  (Falseで全GPU処理・大幅高速化)')

In [ ]:
# ============================================================
# セル 8 置き換え — ONNX エクスポート (FP画像CNN版)
# ============================================================
def export_persona_onnx(policy, persona_id, rp):
    policy.eval()
    policy_cpu = policy.to('cpu')

    # DINOv2 frozen部分は除き、FC+Actorヘッドだけexport
    # HTML側では別途DINOv2相当の特徴量を渡す
    # DINOv2 frozen部分を除き FC+Actor だけ export
    class FCActorOnly(nn.Module):
        def __init__(self, p): super().__init__(); self.fc=p.fc; self.actor=p.actor
        def forward(self, feat): return self.actor(self.fc(feat))

    actor = FCActorOnly(policy_cpu); actor.eval()
    tmp_path  = f'/tmp/persona_{persona_id}_tmp.onnx'
    onnx_path = f'{ONNX_DIR}/persona_{persona_id}.onnx'

    # フラット形式でexport (HTMLでのfetch推論に対応)
    dummy_flat = torch.zeros(1, DINO_DIM)  # CLS(384) のみ

    with torch.no_grad():
        torch.onnx.export(
            actor, dummy_flat, tmp_path,
            input_names=['state'], output_names=['logits'],
            dynamic_axes={'state':{0:'batch'}, 'logits':{0:'batch'}},
            opset_version=17,
        )

    onnx_model = onnx.load(tmp_path)
    onnx.save(onnx_model, onnx_path, save_as_external_data=False)
    for f in [tmp_path, tmp_path+'.data']:
        if os.path.exists(f): os.remove(f)

    size_mb = os.path.getsize(onnx_path)/1e6
    print(f'  ✓ persona_{persona_id}.onnx  ({size_mb:.1f}MB)')

    import onnxruntime as ort
    sess = ort.InferenceSession(onnx_path, providers=['CPUExecutionProvider'])
    dummy_np = np.zeros((1, DINO_DIM), dtype=np.float32)
    logits = sess.run(None, {'state': dummy_np})[0]
    print(f'  ✓ 推論テスト OK  logits={logits[0].round(3)}')

    meta = {
        'persona_id':   persona_id,
        'persona_name': rp.get('persona_name', f'Persona {persona_id}'),
        'description':  rp.get('description', ''),
        'reward_params': rp,
        'obs_type':       'fp_image',
        'input_size':     DINO_DIM,            # CLS(384) のみ
        'obs_components': ['cls'],
        'n_bldg_classes': N_BLDG_CLASSES,
        'bldg_classes':   BLDG_CLASSES,
        'img_w': IMG_W, 'img_h': IMG_H, 'img_ch': IMG_CH,
        'fov_deg': 60.0, 'ray_max': RAY_MAX,
        'output_size':  ACT_DIM,
        'actions':      ['forward', 'turn_left', 'turn_right'],
        'move_dist':    MOVE_DIST, 'rot_deg': 20.0,
        'grid_size':    GRID,
        'map':          MAP_NP.tolist(),
        'building_cells': BCELLS_NP.tolist(),
        'cell_rgb': [
            [12,30,74], [176,180,172], [196,32,32], [35,104,40]
        ],
        'sky_rgb':   [6,12,20],
        'floor_rgb': [26,40,32],
    }
    meta_path = onnx_path.replace('.onnx', '_meta.json')
    with open(meta_path, 'w', encoding='utf-8') as f:
        json.dump(meta, f, indent=2, ensure_ascii=False)
    print(f'  ✓ persona_{persona_id}_meta.json 保存')

    policy.to(DEVICE)
    return onnx_path

print('export_persona_onnx (FP-CNN版) ✓')

In [ ]:
# ============================================================
# 追加セル — DINOv2 本体を cls+patch 出力で ONNX エクスポート
# (ImageNet正規化をグラフ内に内蔵。入力は [0,1] RGB 画像)
# 出力: data/dinov2_vits14.onnx — サーバーが全ペルソナで共有する
# ============================================================
class DinoFeat(nn.Module):
    def __init__(self, backbone, mean, std):
        super().__init__()
        self.backbone = backbone
        self.register_buffer('mean', torch.tensor(mean).view(1,3,1,1))
        self.register_buffer('std',  torch.tensor(std).view(1,3,1,1))
    def forward(self, img):                     # (N,3,224,224) RGB [0,1]
        x = (img - self.mean) / self.std
        out = self.backbone.forward_features(x)
        return out['x_norm_clstoken'], out['x_norm_patchtokens']

_dino = DinoFeat(dino_backbone, DINO_MEAN, DINO_STD).to(DEVICE).eval()
_dummy_img = torch.zeros(1, 3, IMG_H, IMG_W, device=DEVICE)
_dino_path = f'{ONNX_DIR}/dinov2_vits14.onnx'
torch.onnx.export(_dino, _dummy_img, _dino_path,
    input_names=['image'], output_names=['cls','patch'],
    dynamic_axes={'image':{0:'batch'}, 'cls':{0:'batch'}, 'patch':{0:'batch'}},
    opset_version=17)
print(f'✓ dinov2_vits14.onnx エクスポート: {os.path.getsize(_dino_path)/1e6:.1f}MB')

import onnxruntime as _ortc
_s = _ortc.InferenceSession(_dino_path, providers=['CPUExecutionProvider'])
_o = _s.run(None, {'image': np.zeros((1,3,IMG_H,IMG_W), np.float32)})
print('  cls', _o[0].shape, ' patch', _o[1].shape)   # (1,384) (1,256,384)


In [ ]:
# ============================================================
# セル 9 置き換え — train_persona (FP画像CNN版)
# バッファを (ROLLOUT, N, 3, H, W) の5次元に変更
# ============================================================
MAP_RANDOMIZE_EVERY = 20

def train_persona(persona_id, rp, total_steps=STEPS_PER_PERSONA):
    print('='*60)
    print(f'学習開始: [{persona_id}] {rp["persona_name"]}')
    print(f'  観測: FP画像 {IMG_W}×{IMG_H}×{IMG_CH}ch')
    print(f'  目標: {total_steps/1e6:.0f}M steps')
    dr_str = f'{MAP_RANDOMIZE_EVERY} update ごと' if MAP_RANDOMIZE_EVERY > 0 else '無効'
    print(f'  Domain Randomization: {dr_str}')
    print('='*60)

    policy    = PolicyNet().to(DEVICE)
    optimizer = torch.optim.Adam(policy.parameters(), lr=LR)
    # ペルソナ別ハイパラ (性格差を重みに焼き付ける)。無ければグローバル既定。
    gm = float(rp.get('gamma', GAMMA)); ec = float(rp.get('ent_coef', ENT_COEF))
    print(f'  hp: ent_coef={ec}  gamma={gm}  action_repeat={ACTION_REPEAT}')
    # policy_ref を渡すことで環境がセグメンテーション判定を使用できる
    env       = PersonaVecEnv(N_ENVS, rp, policy_ref=policy)

    ckpt_path    = f'{SAVE_DIR}/ckpt_{persona_id}.pt'
    start_update = 0
    log = {'reward':[], 'trips':[], 'viols':[], 'explore':[],
           'gpu_util':[], 'gpu_vram':[], 'map_changes':[]}

    if os.path.exists(ckpt_path):
        ck = torch.load(ckpt_path, map_location=DEVICE)
        policy.load_state_dict(ck['policy'])
        optimizer.load_state_dict(ck['optimizer'])
        start_update = ck.get('update', 0)
        log = ck.get('log', log)
        print(f'  ✓ 再開: update={start_update}')

    obs = env.reset_all()  # (N, 3, H, W)

    # ── バッファ: DINOv2 CLS特徴(384)をキャッシュ (画像は溜めない=メモリ激減) ──
    feat_buf = torch.zeros(ROLLOUT, N_ENVS, DINO_DIM, device=DEVICE)
    act_buf  = torch.zeros(ROLLOUT, N_ENVS, dtype=torch.long, device=DEVICE)
    rew_buf  = torch.zeros(ROLLOUT, N_ENVS, device=DEVICE)
    done_buf = torch.zeros(ROLLOUT, N_ENVS, device=DEVICE)
    logp_buf = torch.zeros(ROLLOUT, N_ENVS, device=DEVICE)
    val_buf  = torch.zeros(ROLLOUT, N_ENVS, device=DEVICE)

    total = start_update * B
    r_s = t_s = v_s = e_s = 0.0
    t0  = time.time()
    LOG_EVERY   = 10
    map_changes = 0

    for update in range(start_update, total_steps // B + 1):

        # Domain Randomization
        if MAP_RANDOMIZE_EVERY > 0 and update > 0 and update % MAP_RANDOMIZE_EVERY == 0:
            randomize_map()
            env.visited.zero_()
            obs = env.reset_all()
            map_changes += 1

        # Rollout
        for t in range(ROLLOUT):
            with torch.no_grad():
                feat = policy.feat(obs)                 # DINOv2(frozen, チャンク)
                actions, logps, _, values = policy.act_feat(feat)
            feat_buf[t] = feat
            act_buf[t]  = actions
            logp_buf[t] = logps
            val_buf[t]  = values
            obs, rew, done = env.step(actions)
            rew_buf[t]  = rew
            done_buf[t] = done.float()
            total += N_ENVS
            # GPUテンソルのまま加算 (毎ステップの .item() 同期を排除)
            r_s += rew.sum()
            t_s += (rew >= rp['goal_reward']*0.8).sum()
            v_s += (rew <= -rp['violation_penalty']*0.8).sum()
            e_s += (rew >= rp['explore_bonus']*0.8).sum()

        # GAE
        with torch.no_grad():
            _, lv = policy.head(policy.feat(obs)); lv = lv.squeeze(-1)
        adv = torch.zeros_like(rew_buf)
        gae = torch.zeros(N_ENVS, device=DEVICE)
        for t in reversed(range(ROLLOUT)):
            nv    = lv if t==ROLLOUT-1 else val_buf[t+1]
            delta = rew_buf[t] + gm*nv*(1-done_buf[t]) - val_buf[t]
            gae   = delta + gm*GAE_LAM*(1-done_buf[t])*gae
            adv[t] = gae
        ret = adv + val_buf

        # PPO — reshape で5次元→4次元 (B, C, H, W)
        feat_f = feat_buf.reshape(B, DINO_DIM)
        act_f  = act_buf.reshape(B)
        logp_f = logp_buf.reshape(B)
        adv_f  = adv.reshape(B)
        ret_f  = ret.reshape(B)
        adv_f  = (adv_f - adv_f.mean()) / (adv_f.std() + 1e-8)

        for _ in range(EPOCHS):
            perm = torch.randperm(B, device=DEVICE)
            for s in range(0, B, MINIBATCH):
                mb     = perm[s:s+MINIBATCH]
                lg, vs = policy.head(feat_f[mb])
                dist   = torch.distributions.Categorical(torch.softmax(lg,-1))
                nlp    = dist.log_prob(act_f[mb])
                ent    = dist.entropy().mean()
                ratio  = torch.exp(nlp - logp_f[mb])
                lpi    = -torch.min(
                    ratio*adv_f[mb],
                    torch.clamp(ratio,1-CLIP_EPS,1+CLIP_EPS)*adv_f[mb]
                ).mean()
                lvf    = (vs.squeeze()-ret_f[mb]).pow(2).mean()
                loss   = lpi + VF_COEF*lvf - ec*ent
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(policy.parameters(), 0.5)
                optimizer.step()

        # ログ
        if update % LOG_EVERY == 0 and update > 0:
            d  = LOG_EVERY * B
            ar = float(r_s)/d; at = float(t_s)/d*100; av = float(v_s)/d*100; ae = float(e_s)/d*100
            gpu_util, gpu_used, _ = get_gpu_stats()
            sps = LOG_EVERY*B/(time.time()-t0)
            log['reward'].append(round(float(ar),4))
            log['trips'].append(round(float(at),3))
            log['viols'].append(round(float(av),3))
            log['explore'].append(round(float(ae),3))
            log['gpu_util'].append(round(float(gpu_util),1))
            log['gpu_vram'].append(round(float(gpu_used),3))
            log['map_changes'].append(map_changes)
            r_s=t_s=v_s=e_s=0.0; t0=time.time()
            clear_output(wait=True)

            fig = plt.figure(figsize=(22,3.5))
            fig.patch.set_facecolor('#0a0c10')
            for i,(k,c,title) in enumerate([
                ('reward','#00ddb4','Reward/step'),
                ('trips','#ffc840','GoalRate%'),
                ('viols','#ff5050','ViolRate%'),
                ('explore','#aa88ff','ExploreRate%'),
                ('gpu_util','#44aaff','GPU%'),
            ]):
                ax=fig.add_subplot(1,6,i+1); ax.set_facecolor('#12161e')
                v=log[k]
                if len(v)>=2: ax.plot(v,color=c,lw=1.5)
                ax.set_title(f'{title}\n{v[-1]:.2f}' if v else title,
                             color='#c8d8e8',fontsize=8)
                ax.grid(color='#1e2530',lw=0.5); ax.spines[:].set_color('#1e2530')
                if k=='gpu_util': ax.set_ylim(0,100)
            # 現在のFPサンプル画像
            ax_img=fig.add_subplot(1,6,6)
            with torch.no_grad():
                sample=render_fp_batch(
                    env.x[:1],env.y[:1],env.th[:1],
                    MAP_T,CELL_RGB_GPU,SKY_RGB_GPU,FLOOR_RGB_GPU,
                    RAY_ANGLES_GPU,RAY_DISTS_GPU,
                    IMG_W,IMG_H,GRID,RAY_MAX
                ).cpu()
            ax_img.imshow(sample[0].permute(1,2,0).numpy())
            ax_img.set_title('現在のFP観測', color='#c8d8e8', fontsize=8)
            ax_img.axis('off')
            fig.suptitle(
                f'[{persona_id}] {rp["persona_name"]}  |  '
                f'Update:{update:,}  Steps:{total/1e6:.1f}M  '
                f'({total/total_steps*100:.1f}%)  Maps:{map_changes}',
                color='#00ddb4', fontsize=11)
            plt.tight_layout(); plt.show()
            print(f'[{persona_id}] Upd:{update:5d} | R:{ar:7.4f} | '
                  f'Goal:{at:.2f}% | Viol:{av:.2f}% | Exp:{ae:.2f}% | '
                  f'{sps:.0f}sps | GPU:{gpu_util:.0f}% VRAM:{gpu_used:.2f}GB')

        if update%50==0 and update>0:
            torch.save({'policy':policy.state_dict(),
                        'optimizer':optimizer.state_dict(),
                        'update':update,'log':log,
                        'map_changes':map_changes},ckpt_path)
            print(f'  💾 [{persona_id}] ckpt 保存 (update={update})')

    print(f'\n[{persona_id}] 学習完了 ({total/1e6:.1f}M steps)')
    return policy, log

print('train_persona (FP-CNN版) ✓')

In [ ]:
# ============================================================
# 追加セル — 目標条件付け + 補助観測 (compass / 訪問メモリ / social) パッチ
# ------------------------------------------------------------
# ポリシー入力を [CLS(384), z(8), aux(9)] = 401 に拡張する。
#
#   z(8)   : 行き先建物タイプ one-hot。★目的地建物 (gx,gy) の実タイプと連動★
#            (旧版は z とゴールセルが無関係で、意味のない条件付けだった)。
#            GOAL_NONE_PROB の割合で z=ゼロ (目標タイプなし) を混ぜる。
#   aux(9) : エージェント自身の状態から計算する補助特徴。
#     compass(3) : 目的地の相対方位 sin/cos + 距離/GRID
#                  → 旧版は「観測できない目的地」への距離報酬・到着報酬 (goal_reward
#                    10〜30) が学習不能ノイズになり、ペルソナ差が消える主因だった
#     visited(4) : 前/左/右/後 4セクタ (半径 VISIT_R) の訪問済みセル率
#                  → 「経路の記憶がなく同じ場所を巡回する」問題への対処。
#                    explore_bonus / revisit_penalty がこれで初めて学習可能になる
#     social(2)  : パートナー(同一マップ上の他エージェント)の相対方位 sin/cos × 近接度
#                  → social_bonus を学習可能にする (ペルソナC)
#
# server.js / standalone/index.html は meta の aux_dim / obs_components を見て
# 同一レイアウト・同一式で aux を組み立てる (buildAux 関数)。
# 注意: fc 入力が 392→401 に変わるため旧 ckpt_*.pt は読めない (自動で新規学習)。
# ============================================================
GOAL_DIM         = N_BLDG_CLASSES   # = 8 (建物タイプ one-hot)
GOAL_NONE_PROB   = 0.4              # z=ゼロ(目標タイプなし) で学習するエピソード割合
GOAL_REACH_BONUS = 0.3              # 目標タイプが視界中央 かつ 移動中 のときの dense 報酬
                                    # (moved ゲートで「止まって凝視して稼ぐ」退化を防ぐ)

AUX_COMPASS = 3; AUX_VISIT = 4; AUX_SOCIAL = 2
AUX_DIM  = AUX_COMPASS + AUX_VISIT + AUX_SOCIAL     # 9
HEAD_IN  = DINO_DIM + GOAL_DIM + AUX_DIM            # 401
VISIT_R      = 5      # 訪問メモリの参照半径 (セル)
SOCIAL_RANGE = 8.0    # social 近接度が 0 になる距離
print(f'[goal+aux] HEAD_IN={HEAD_IN} = CLS({DINO_DIM}) + z({GOAL_DIM}) + aux({AUX_DIM})')

# 訪問セクタ用オフセット (定数)
_vr = torch.arange(-VISIT_R, VISIT_R + 1, device=DEVICE)
_DR, _DC = torch.meshgrid(_vr, _vr, indexing='ij')
_DR = _DR.reshape(-1); _DC = _DC.reshape(-1)                       # (K,) K=121
_OFF_ANG = torch.atan2(_DC.float(), _DR.float())                   # マップ座標系の方位

def _wrap(a):
    """角度を (-π, π] に正規化"""
    return torch.atan2(torch.sin(a), torch.cos(a))


# ── PolicyNet: fc 入力を HEAD_IN に拡張 ──────────────────────────────────────
class PolicyNetGoal(PolicyNet):
    def __init__(self):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(HEAD_IN, 256),
            nn.LayerNorm(256),
            nn.ReLU(),
            nn.Linear(256, 256),
            nn.ReLU(),
        )

    def head(self, feat):
        """feat = [CLS(384), z(8), aux(9)] = (N, 401) → logits / value"""
        h = self.fc(feat)
        return self.actor(h), self.critic(h)

    def act_feat(self, feat):
        lg, val = self.head(feat)
        dist = torch.distributions.Categorical(torch.softmax(lg, -1))
        a = dist.sample()
        return a, dist.log_prob(a), dist.entropy(), val.squeeze(-1)

    def forward(self, x, goal=None, aux=None):
        f = self._cls(x)
        if goal is None:
            goal = torch.zeros(f.shape[0], GOAL_DIM, device=f.device)
        if aux is None:
            aux = torch.zeros(f.shape[0], AUX_DIM, device=f.device)
        return self.head(torch.cat([f, goal, aux], -1))

PolicyNet = PolicyNetGoal


# ── PersonaVecEnv: z を目的地建物タイプに連動 + aux 観測 ─────────────────────
class PersonaVecEnvGoal(PersonaVecEnv):
    def __init__(self, n, rp, policy_ref=None):
        self.goal_type = None
        super().__init__(n, rp, policy_ref=policy_ref)   # __init__ 内で _rst が呼ばれる

    def _refresh_goal(self):
        """z(one-hot) を「現在の目的地建物 (gx,gy) のタイプ」と同期する。
           z_none エピソードは z=ゼロのまま (compass は常に有効)。"""
        bt = BTYPE_T[self.gx.long().clamp(0, GRID - 1),
                     self.gy.long().clamp(0, GRID - 1)].clamp(min=0)
        self.goal_type = torch.where(self.z_none, torch.full_like(bt, -1), bt)
        oh = torch.zeros(self.n, GOAL_DIM, device=DEVICE)
        hg = self.goal_type >= 0
        if bool(hg.any()):
            idx = torch.nonzero(hg).squeeze(1)
            oh[idx, self.goal_type[idx]] = 1.0
        self.goal = oh

    def _rst(self, mask):
        if self.goal_type is None:   # 初回 (__init__ 経由)
            self.goal_type = torch.full((self.n,), -1, dtype=torch.long, device=DEVICE)
            self.goal      = torch.zeros(self.n, GOAL_DIM, device=DEVICE)
            self.z_none    = torch.ones(self.n, dtype=torch.bool, device=DEVICE)
        super()._rst(mask)
        nz = torch.rand(self.n, device=DEVICE) < GOAL_NONE_PROB
        self.z_none = torch.where(mask, nz, self.z_none)
        self._refresh_goal()

    def aux(self):
        """補助観測 (N, AUX_DIM=9)。server.js / index.html の buildAux() と
           同一レイアウト・同一式にすること (順序: compass3, visited4, social2)。"""
        N = self.n
        # compass(3): 目的地の相対方位 sin/cos + 距離/GRID
        dx = self.gx - self.x; dy = self.gy - self.y
        d  = torch.sqrt(dx * dx + dy * dy)
        b  = _wrap(torch.atan2(dy, dx) - self.th)
        compass = torch.stack([torch.sin(b), torch.cos(b),
                               (d / GRID).clamp(max=1.0)], -1)
        # visited(4): 前/左/右/後セクタの訪問済み率 (マップ範囲外は訪問済み扱い)
        r0 = self.x.long().clamp(0, GRID - 1); c0 = self.y.long().clamp(0, GRID - 1)
        rr = r0[:, None] + _DR[None, :]; cc = c0[:, None] + _DC[None, :]
        inb = (rr >= 0) & (rr < GRID) & (cc >= 0) & (cc < GRID)
        vis = self.visited[torch.arange(N, device=DEVICE)[:, None],
                           rr.clamp(0, GRID - 1), cc.clamp(0, GRID - 1)].bool()
        vis = vis | ~inb
        rel = _wrap(_OFF_ANG[None, :] - self.th[:, None])          # (N, K)
        sec_front = rel.abs() <= math.pi / 4
        sec_left  = (rel < -math.pi / 4) & (rel > -3 * math.pi / 4)
        sec_right = (rel >  math.pi / 4) & (rel <  3 * math.pi / 4)
        sec_back  = ~(sec_front | sec_left | sec_right)
        vs = []
        for m in (sec_front, sec_left, sec_right, sec_back):
            cnt = m.float().sum(-1).clamp(min=1.0)
            vs.append((vis & m).float().sum(-1) / cnt)
        visit4 = torch.stack(vs, -1)
        # social(2): パートナー (roll1 = 同一マップ上の別エージェント) の方位×近接度
        pxo = torch.roll(self.x, 1); pyo = torch.roll(self.y, 1)
        sdx = pxo - self.x; sdy = pyo - self.y
        sd  = torch.sqrt(sdx * sdx + sdy * sdy)
        sb  = _wrap(torch.atan2(sdy, sdx) - self.th)
        prox = (1.0 - sd / SOCIAL_RANGE).clamp(0.0, 1.0)
        social = torch.stack([torch.sin(sb) * prox, torch.cos(sb) * prox], -1)
        return torch.cat([compass, visit4, social], -1)

    def step(self, actions):
        obs, rew, done = super().step(actions)
        # 目標タイプが視界中央 かつ 実際に移動した ときだけ dense 報酬
        # (moved ゲートが無いと「目標建物を凝視して停止」が最適解になり得る)
        hg = self.goal_type >= 0
        if bool(hg.any()):
            iv    = building_in_view(self.x, self.y, self.th)
            seen  = iv.gather(1, self.goal_type.clamp(min=0)[:, None]).squeeze(1) > 0.5
            moved = (torch.abs(self.x - self.px) + torch.abs(self.y - self.py)) > 0.05
            gvr = torch.where(hg & seen & moved,
                              torch.full((self.n,), GOAL_REACH_BONUS, device=DEVICE),
                              torch.zeros(self.n, device=DEVICE))
        else:
            gvr = torch.zeros(self.n, device=DEVICE)
        self._last_gvr = gvr
        rew = rew + gvr
        # 到着で目的地が変わった env の z を同期 (done リセット分は _rst 内で同期済み)
        self._refresh_goal()
        return obs, rew, done

PersonaVecEnv = PersonaVecEnvGoal


# ── ONNX エクスポート: ヘッド入力 401 + aux レイアウトを meta に記録 ──────────
def export_persona_onnx(policy, persona_id, rp):
    policy.eval()
    policy_cpu = policy.to('cpu')

    class FCActorOnly(nn.Module):
        def __init__(self, p): super().__init__(); self.fc = p.fc; self.actor = p.actor
        def forward(self, feat): return self.actor(self.fc(feat))

    actor = FCActorOnly(policy_cpu); actor.eval()
    tmp_path  = f'/tmp/persona_{persona_id}_tmp.onnx'
    onnx_path = f'{ONNX_DIR}/persona_{persona_id}.onnx'
    dummy_flat = torch.zeros(1, HEAD_IN)   # [CLS(384), z(8), aux(9)]

    with torch.no_grad():
        torch.onnx.export(
            actor, dummy_flat, tmp_path,
            input_names=['state'], output_names=['logits'],
            dynamic_axes={'state': {0: 'batch'}, 'logits': {0: 'batch'}},
            opset_version=17,
        )
    onnx.save(onnx.load(tmp_path), onnx_path, save_as_external_data=False)
    for f in [tmp_path, tmp_path + '.data']:
        if os.path.exists(f): os.remove(f)

    import onnxruntime as ort
    sess   = ort.InferenceSession(onnx_path, providers=['CPUExecutionProvider'])
    logits = sess.run(None, {'state': np.zeros((1, HEAD_IN), np.float32)})[0]
    print(f'  ✓ persona_{persona_id}.onnx  ({os.path.getsize(onnx_path)/1e6:.1f}MB)  '
          f'in={HEAD_IN}  logits={logits[0].round(3)}')

    meta = {
        'persona_id':     persona_id,
        'persona_name':   rp.get('persona_name', f'Persona {persona_id}'),
        'description':    rp.get('description', ''),
        'reward_params':  rp,
        'obs_type':       'fp_image',
        'input_size':     HEAD_IN,                  # 401 = CLS + z + aux
        'obs_components': ['cls', 'goal', 'compass', 'visited', 'social'],
        'cls_dim':        DINO_DIM,
        'goal_dim':       GOAL_DIM,
        'aux_dim':        AUX_DIM,                  # server.js が buildAux を有効化する目印
        'aux_layout':     {'compass': AUX_COMPASS, 'visited': AUX_VISIT, 'social': AUX_SOCIAL},
        'visit_radius':   VISIT_R,
        'visit_window_ticks': MAX_STEPS * ACTION_REPEAT,  # 本番側の訪問メモリ保持ティック数
        'social_range':   SOCIAL_RANGE,
        'goal_classes':   BLDG_CLASSES,
        'n_bldg_classes': N_BLDG_CLASSES,
        'bldg_classes':   BLDG_CLASSES,
        'img_w': IMG_W, 'img_h': IMG_H, 'img_ch': IMG_CH,
        'fov_deg': 60.0, 'ray_max': RAY_MAX,
        'output_size': ACT_DIM,
        'actions': ['forward', 'turn_left', 'turn_right'],
        'move_dist': MOVE_DIST,
        'rot_deg':          ROT_DECISION_DEG / ACTION_REPEAT,  # per tick (legacy 互換フィールド)
        'rot_per_tick_deg': ROT_DECISION_DEG / ACTION_REPEAT,
        'rot_decision_deg': ROT_DECISION_DEG,
        'action_repeat':    ACTION_REPEAT,
        'grid_size': GRID,
        'map': MAP_NP.tolist(),
        'building_cells': BCELLS_NP.tolist(),
        'cell_rgb': [[12,30,74], [176,180,172], [196,32,32], [35,104,40]],
        'sky_rgb':   [6,12,20],
        'floor_rgb': [26,40,32],
    }
    with open(onnx_path.replace('.onnx', '_meta.json'), 'w', encoding='utf-8') as f:
        json.dump(meta, f, indent=2, ensure_ascii=False)
    print(f'  ✓ persona_{persona_id}_meta.json 保存 (goal_dim={GOAL_DIM} aux_dim={AUX_DIM})')

    policy.to(DEVICE)
    return onnx_path


# ── train_persona: [CLS, z, aux] を連結して PPO 学習 ─────────────────────────
def train_persona(persona_id, rp, total_steps=STEPS_PER_PERSONA):
    print('=' * 60)
    print(f'学習開始(goal+aux条件付き): [{persona_id}] {rp["persona_name"]}')
    print(f'  入力: CLS({DINO_DIM}) + z({GOAL_DIM}) + aux({AUX_DIM}) = {HEAD_IN}'
          f'  / 目標タイプなし割合={GOAL_NONE_PROB}')
    print('=' * 60)
    gm = float(rp.get('gamma', GAMMA)); ec = float(rp.get('ent_coef', ENT_COEF))
    print(f'  hp: ent_coef={ec}  gamma={gm}  action_repeat={ACTION_REPEAT}  '
          f'approach_coef={rp.get("approach_coef", 0.5):.2f}')

    policy    = PolicyNet().to(DEVICE)
    optimizer = torch.optim.Adam(policy.parameters(), lr=LR)
    env       = PersonaVecEnv(N_ENVS, rp, policy_ref=policy)

    ckpt_path    = f'{SAVE_DIR}/ckpt_{persona_id}.pt'
    start_update = 0
    log = {'reward': [], 'goal_view': []}
    if os.path.exists(ckpt_path):
        try:
            ck = torch.load(ckpt_path, map_location=DEVICE)
            policy.load_state_dict(ck['policy'])
            optimizer.load_state_dict(ck['optimizer'])
            start_update = ck.get('update', 0)
            log = ck.get('log', log)
            print(f'  ✓ 再開: update={start_update}')
        except Exception as e:
            print(f'  ⚠ 旧 ckpt と次元不一致のため新規学習: {e}')

    obs = env.reset_all()
    feat_buf = torch.zeros(ROLLOUT, N_ENVS, HEAD_IN, device=DEVICE)
    act_buf  = torch.zeros(ROLLOUT, N_ENVS, dtype=torch.long, device=DEVICE)
    rew_buf  = torch.zeros(ROLLOUT, N_ENVS, device=DEVICE)
    done_buf = torch.zeros(ROLLOUT, N_ENVS, device=DEVICE)
    logp_buf = torch.zeros(ROLLOUT, N_ENVS, device=DEVICE)
    val_buf  = torch.zeros(ROLLOUT, N_ENVS, device=DEVICE)

    total = start_update * B
    r_s = g_s = 0.0
    t0 = time.time()
    LOG_EVERY = 10

    for update in range(start_update, total_steps // B + 1):
        if MAP_RANDOMIZE_EVERY > 0 and update > 0 and update % MAP_RANDOMIZE_EVERY == 0:
            randomize_map(); env.visited.zero_(); obs = env.reset_all()

        for t in range(ROLLOUT):
            with torch.no_grad():
                cls  = policy.feat(obs)                                  # (N, 384)
                feat = torch.cat([cls, env.goal, env.aux()], -1)         # (N, 401)
                actions, logps, _, values = policy.act_feat(feat)
            feat_buf[t] = feat
            act_buf[t]  = actions
            logp_buf[t] = logps
            val_buf[t]  = values
            obs, rew, done = env.step(actions)
            rew_buf[t]  = rew
            done_buf[t] = done.float()
            total += N_ENVS
            r_s += rew.sum(); g_s += env._last_gvr.sum()

        with torch.no_grad():
            last = torch.cat([policy.feat(obs), env.goal, env.aux()], -1)
            _, lv = policy.head(last); lv = lv.squeeze(-1)
        adv = torch.zeros_like(rew_buf)
        gae = torch.zeros(N_ENVS, device=DEVICE)
        for t in reversed(range(ROLLOUT)):
            nv    = lv if t == ROLLOUT - 1 else val_buf[t + 1]
            delta = rew_buf[t] + gm * nv * (1 - done_buf[t]) - val_buf[t]
            gae   = delta + gm * GAE_LAM * (1 - done_buf[t]) * gae
            adv[t] = gae
        ret = adv + val_buf

        feat_f = feat_buf.reshape(B, HEAD_IN)
        act_f  = act_buf.reshape(B)
        logp_f = logp_buf.reshape(B)
        adv_f  = adv.reshape(B)
        ret_f  = ret.reshape(B)
        adv_f  = (adv_f - adv_f.mean()) / (adv_f.std() + 1e-8)

        for _ in range(EPOCHS):
            perm = torch.randperm(B, device=DEVICE)
            for s in range(0, B, MINIBATCH):
                mb     = perm[s:s + MINIBATCH]
                lg, vs = policy.head(feat_f[mb])
                dist   = torch.distributions.Categorical(torch.softmax(lg, -1))
                nlp    = dist.log_prob(act_f[mb])
                ent    = dist.entropy().mean()
                ratio  = torch.exp(nlp - logp_f[mb])
                lpi    = -torch.min(
                    ratio * adv_f[mb],
                    torch.clamp(ratio, 1 - CLIP_EPS, 1 + CLIP_EPS) * adv_f[mb]
                ).mean()
                lvf    = (vs.squeeze() - ret_f[mb]).pow(2).mean()
                loss   = lpi + VF_COEF * lvf - ec * ent
                optimizer.zero_grad(); loss.backward()
                torch.nn.utils.clip_grad_norm_(policy.parameters(), 0.5)
                optimizer.step()

        if update % LOG_EVERY == 0 and update > 0:
            d = LOG_EVERY * B
            ar = float(r_s) / d; ag = float(g_s) / d
            sps = LOG_EVERY * B / (time.time() - t0)
            gpu_util, gpu_used, _ = get_gpu_stats()
            log['reward'].append(round(ar, 4)); log['goal_view'].append(round(ag, 4))
            r_s = g_s = 0.0; t0 = time.time()
            print(f'[{persona_id}] Upd:{update:5d} Steps:{total/1e6:5.1f}M | '
                  f'R:{ar:7.4f} | goalView:{ag:.4f} | {sps:.0f}sps GPU:{gpu_util:.0f}%')

        if update % 50 == 0 and update > 0:
            torch.save({'policy': policy.state_dict(),
                        'optimizer': optimizer.state_dict(),
                        'update': update, 'log': log}, ckpt_path)
            print(f'  💾 [{persona_id}] ckpt 保存 (update={update})')

    print(f'\n[{persona_id}] 学習完了 ({total/1e6:.1f}M steps)')
    return policy, log

print('✓ goal+aux パッチ適用: PolicyNet / PersonaVecEnv / export / train を上書き')
print(f'  観測: CLS({DINO_DIM}) + z({GOAL_DIM}) + compass(3) + visited(4) + social(2) = {HEAD_IN}')


In [ ]:
# ============================================================
# 追加セル — persona_rewards.json を Drive に直接生成
# ------------------------------------------------------------
# Step1 (step1_persona_reward_gen.ipynb) を実行していない場合の代替。
# data/persona_*_meta.json の reward_params を復元した内容と同一。
# 既に SAVE_DIR に persona_rewards.json がある場合は上書きしたくなければ
# このセルをスキップしてよい。
# ============================================================
import json, os

os.makedirs(SAVE_DIR, exist_ok=True)

persona_rewards = {
    "A": {"persona_id": "A", "persona_name": "探索者タロウ",
          "description": "未訪問の場所を積極的に開拓し、同じ道は通りたがらない探索型エージェント。マップ全体を効率よく踏破することを目指す。",
          "explore_bonus": 4.5, "building_bonus": 0.1, "road_bonus": 0.4, "forward_bias": 0.3,
          "revisit_penalty": 1.8, "violation_penalty": 3.0, "goal_reward": 10.0,
          "step_penalty": 0.05, "social_bonus": 0.0, "stall_penalty": 1.5,
          "food_bonus": 0.0, "food_classes": []},
    "B": {"persona_id": "B", "persona_name": "インドア花子",
          "description": "外出を最小限にして目的建物へ最短で移動する慎重派。一度通った安全な道を繰り返し使い、寄り道をしない。",
          "explore_bonus": 0.1, "building_bonus": 0.5, "road_bonus": 0.3, "forward_bias": 0.6,
          "revisit_penalty": 0.0, "violation_penalty": 5.0, "goal_reward": 28.0,
          "step_penalty": 0.4, "social_bonus": 0.0, "stall_penalty": 2.0,
          "food_bonus": 0.0, "food_classes": []},
    "C": {"persona_id": "C", "persona_name": "社交家ケンジ",
          "description": "他のエージェントの近くにいることを好み、人が集まる建物周辺を好んで移動する。一人でいると不安になる社交型エージェント。",
          "explore_bonus": 0.5, "building_bonus": 1.2, "road_bonus": 0.2, "forward_bias": 0.2,
          "revisit_penalty": 0.0, "violation_penalty": 3.0, "goal_reward": 15.0,
          "step_penalty": 0.08, "social_bonus": 3.0, "stall_penalty": 0.3,
          "food_bonus": 0.0, "food_classes": []},
    "D": {"persona_id": "D", "persona_name": "ビジネスマン誠",
          "description": "常に最短経路で建物から建物へ直進する効率重視型。寄り道や探索は一切せず、ステップ数を最小化することを優先する。",
          "explore_bonus": 0.0, "building_bonus": 0.3, "road_bonus": 0.2, "forward_bias": 0.9,
          "revisit_penalty": 0.2, "violation_penalty": 4.0, "goal_reward": 30.0,
          "step_penalty": 0.5, "social_bonus": 0.0, "stall_penalty": 2.0,
          "food_bonus": 0.0, "food_classes": []},
    "E": {"persona_id": "E", "persona_name": "観光客サラ",
          "description": "建物に立ち寄ることを楽しみ、各建物をじっくり見て回る観光型エージェント。道に迷いながらも建物発見の喜びを行動原理にしている。",
          "explore_bonus": 2.0, "building_bonus": 2.0, "road_bonus": 0.1, "forward_bias": 0.1,
          "revisit_penalty": 0.3, "violation_penalty": 2.5, "goal_reward": 20.0,
          "step_penalty": 0.02, "social_bonus": 0.5, "stall_penalty": 0.1,
          "food_bonus": 0.0, "food_classes": []},
}

_path = f'{SAVE_DIR}/persona_rewards.json'
with open(_path, 'w', encoding='utf-8') as f:
    json.dump(persona_rewards, f, ensure_ascii=False, indent=2)
print(f'✓ persona_rewards.json 作成: {_path}  ({len(persona_rewards)} personas)')


In [ ]:
# ============================================================
# セル 10: 報酬パラメータ読み込み
# ============================================================
rewards_path = f'{SAVE_DIR}/persona_rewards.json'
assert os.path.exists(rewards_path), (
    f'❌ {rewards_path} が見つかりません。\n'
    f'   step1_persona_reward_gen.ipynb を先に実行するか、\n'
    f'   persona_rewards.json (サンプル) を {SAVE_DIR} にアップロードしてください。'
)

with open(rewards_path, encoding='utf-8') as f:
    all_rewards = json.load(f)

# 数値を float に変換
FLOAT_KEYS = ['explore_bonus','building_bonus','road_bonus','forward_bias',
               'revisit_penalty','violation_penalty','goal_reward',
               'step_penalty','social_bonus','stall_penalty']
for pid,rp in all_rewards.items():
    for k in FLOAT_KEYS:
        rp[k] = float(rp.get(k, 0.0))
    rp['food_bonus']   = float(rp.get('food_bonus', 0.0))
    rp['food_classes'] = [int(x) for x in rp.get('food_classes', [])]

# ── 追加チューニング項目: 歩容 & ペルソナ別 PPO ハイパラ ──
#   step() が参照する offroad_penalty / turn_penalty と、train_persona が
#   参照する ent_coef / gamma を注入する。JSON に無ければペルソナ別既定を使う。
#   (JSON に明記されていればそちらを優先)
PERSONA_HP = {
    # id: 性格に応じた個性づけ
    'A': dict(ent_coef=0.05,  gamma=0.99,  turn_penalty=0.015, offroad_penalty=0.12),  # 探索型: ふらつき許容・寄り道OK
    'B': dict(ent_coef=0.015, gamma=0.995, turn_penalty=0.030, offroad_penalty=0.20),  # 最短型: 迷わず道沿いに
    'C': dict(ent_coef=0.030, gamma=0.990, turn_penalty=0.020, offroad_penalty=0.15),  # 社交型: 標準
    'D': dict(ent_coef=0.010, gamma=0.995, turn_penalty=0.035, offroad_penalty=0.20),  # 効率型: 決定的・直進重視
    'E': dict(ent_coef=0.060, gamma=0.970, turn_penalty=0.010, offroad_penalty=0.10),  # 観光型: 寄り道多い・近視眼的
}
DEFAULT_HP = dict(ent_coef=ENT_COEF, gamma=GAMMA, turn_penalty=0.02, offroad_penalty=0.15)
for pid, rp in all_rewards.items():
    hp = {**DEFAULT_HP, **PERSONA_HP.get(pid, {})}
    for k, v in hp.items():
        rp[k] = float(rp.get(k, v))   # JSON に無い項目のみ既定で補完

# 目的地への接近シェイピング係数 (JSON に無ければ goal_reward 比で自動設定)
# 旧版の固定 +0.5/-0.2 と違いペルソナ差が出る (B/D は強く目的地志向、A/E は弱め)
for pid, rp in all_rewards.items():
    rp['approach_coef'] = float(rp.get('approach_coef', rp['goal_reward'] / 30.0))

print(f'✓ {len(all_rewards)} ペルソナ読み込み完了')
print()
for pid,rp in all_rewards.items():
    print(f'  [{pid}] {rp["persona_name"]}')
    print(f'       explore={rp["explore_bonus"]:.1f}  '
          f'goal={rp["goal_reward"]:.1f}  '
          f'social={rp["social_bonus"]:.1f}  '
          f'step_pen={rp["step_penalty"]:.2f}')
    print(f'       ent={rp["ent_coef"]:.3f}  gamma={rp["gamma"]:.3f}  '
          f'turn_pen={rp["turn_penalty"]:.3f}  offroad_pen={rp["offroad_penalty"]:.2f}  '
          f'approach={rp["approach_coef"]:.2f}')

In [ ]:
# ============================================================
# セル 11: 全ペルソナを順次学習 → ONNX エクスポート
# ============================================================
# 学習するペルソナを選択
# 全ペルソナ: list(all_rewards.keys())
# 途中から:   ['B','C','D','E']  など
TRAIN_PERSONAS = list(all_rewards.keys())

onnx_paths = {}

for pid in TRAIN_PERSONAS:
    rp = all_rewards[pid]

    # 学習
    policy, log = train_persona(pid, rp)

    # ONNX エクスポート
    print(f'\n[{pid}] ONNX エクスポート中...')
    path = export_persona_onnx(policy, pid, rp)
    onnx_paths[pid] = path

    # GPU メモリ解放
    del policy
    torch.cuda.empty_cache()
    print(f'\n→ [{pid}] 完了。次のペルソナへ...\n')
    time.sleep(2)

print('='*60)
print('✓ 全ペルソナの学習・エクスポート完了')
print(f'保存先: マイドライブ > mesa_persona_onnx')
print()
for pid, path in onnx_paths.items():
    size_mb = os.path.getsize(path)/1e6
    print(f'  [{pid}] persona_{pid}.onnx  ({size_mb:.1f}MB)')

In [ ]:
# ============================================================
# セル 12: 最終確認
# ============================================================
print('mesa_persona_onnx フォルダの内容:')
print()
total_mb = 0
for fname in sorted(os.listdir(ONNX_DIR)):
    fpath = f'{ONNX_DIR}/{fname}'
    mb    = os.path.getsize(fpath)/1e6
    total_mb += mb
    has_data = os.path.exists(fpath+'.data')
    flag = '⚠️  外部データあり (ブラウザ非対応)' if has_data else '✓ 単一ファイル'
    print(f'  {fname:<42} {mb:6.1f}MB  {flag}')

print(f'\n  合計: {total_mb:.1f}MB')
print()
print('次のステップ:')
print('  1. Google Drive から .onnx と _meta.json をダウンロード')
print('  2. step3_persona_city_sim.html をブラウザで開く')
print('  3. 各ペルソナの .onnx と _meta.json を同時に選択してロード')
print('  4. Start Simulation !')